# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 07 - Exercises on Graphs

In [ ]:
!gdown 1KfSgY3ae8Gqs89JSrBZHgzL_DbdEl_I2

In [ ]:
def load_data(filename, use_counts=False):
    with open("gen9ou-0.txt", "r") as file:
      lines = file.readlines()
      pokemon_teammates = {}
      special_string = "+----------------------------------------+"
      found_special = True
      next_is_pokemon_name = False
      next_is_teammate = True
      current_pokemon = None
      for line in lines:
          line = line.strip()
          if line == "+----------------------------------------+":
            if found_special:
              next_is_pokemon_name = True
            found_special = True
            next_is_teammate = False
          else:
            if next_is_pokemon_name:
              current_pokemon = "".join(line.split("|")[1]).strip()
              pokemon_teammates[current_pokemon] = {}
              next_is_pokemon_name = False
              found_special = False
            elif next_is_teammate:
              app = "".join(line.split("|")[1:-1]).strip().split(" ")
              teammate_name = " ".join(app[:-1]).strip()
              teammate_prob = float(app[-1].strip()[:-1]) / 100
              if use_counts:
                teammate_prob *= raw_count
              pokemon_teammates[current_pokemon][teammate_name] = teammate_prob
            elif found_special:
              found_special = False
              if "Teammates" in line:
                next_is_teammate = True
              elif "Raw count" in line and use_counts:
                raw_count = int(line.split(": ")[1].split("|")[0].strip())
    return pokemon_teammates

# Extract the data to build the graph.
pokemon_teammates = load_data("gen9ou-0.txt", use_counts=False)

In [ ]:
for key in pokemon_teammates.keys():
  print(key)
  print(pokemon_teammates[key])

## Exercise 1

Iterate over the keys and create a node for each key.

Next, iterate over the sub-keys and create an edge whose weight is the corresponding value: if an edge already exist, add the weight to the current value instead.

Finally, count the number of nodes and edges in the graph.

In [ ]:
nodes = [] # A list should be okay since the keys should be unique
edges = {} # Use a dictionary where the key is a tuple of nodes and the value is the corresponding weight.
for key in pokemon_teammates.keys():
  nodes.append(key)
  neighbours = pokemon_teammates[key]
  for k, v in neighbours.items():
    edge = (key, k)
    edges[edge] = edges.get(edge, 0) + v # Create a new edge or update an existing edge

# Save the graph as a tuple and look at its size.
graph = (nodes, edges)
print(f"Number of Nodes: {len(nodes)}")
print(f"Number of Edges: {len(edges.keys())}")

## Exercise 2

Cut the edges with weight less than $0.2$ and remove any node that remained without edges.

In [ ]:
edges_threshold = {edge: edges[edge] for edge in edges if edges[edge] >= 0.2}
nodes_threshold = []
for node in nodes:
  # Remember that the graph is directed because tuples have been used.
  valid_edges = [edge for edge in edges_threshold.keys() if node in edge]
  if len(valid_edges) > 0:
    nodes_threshold.append(node)

# Study the modified graph.
graph_threshold = (nodes_threshold, edges_threshold)
print(f"Remaining Edges: {len(edges_threshold.keys())}")
print(f"Remaining Nodes: {len(nodes_threshold)}")

## Exercise 3

For each node in the modified graph, compute the in-degree and out-degree and plot the corresponding histograms.

In [ ]:
import matplotlib.pyplot as plt

# Remember that the in-degree of a node counts the edges entering it, whereas the out-degree of a node counts the edges leaving it.
in_degrees = {node: 0 for node in nodes_threshold}
out_degrees = {node: 0 for node in nodes_threshold}
for node in nodes_threshold:
  in_edges = [edge for edge in edges_threshold.keys() if edge[1] == node]
  out_edges = [edge for edge in edges_threshold.keys() if edge[0] == node]
  in_degrees[node] = len(in_edges)
  out_degrees[node] = len(out_edges)

# Analyse the total degrees for the graph.
print(f"Number of Edges: {len(edges_threshold.keys())}")
print(f"Total In-Degree: {sum(in_degrees.values())}")
print(f"Total Out-Degree: {sum(out_degrees.values())}")

# Plot the in-degree histogram.
plt.hist(in_degrees.values(), label="In-Degree", bins=20, alpha=0.8, edgecolor="red")

# Plot the out-degree histogram.
plt.hist(out_degrees.values(), label="Out-Degree", bins=20, alpha=0.7, edgecolor="yellow")

plt.legend()
plt.show()

## Exercise 4

Find all paths of length at most $6$ between any couple of nodes.

Then, find the shortest and longest paths between any pairs of nodes.

In [ ]:
from collections import defaultdict

# Create an adjacency list that, for each node, stores its neighbours and the weight of the corresponding edge.
adjacency_list = defaultdict(list) # Use defaultdict[list] so that each node is assigned an empty list as default value.
for (u, v), w in edges_threshold.items():
  adjacency_list[u].append((v, w))

In [ ]:
import random

def paths_k(edges, k, source, sink):
  if k == 0 and source == sink:
    return [source]
  if k == 0 and source != sink:
    raise ValueError("No paths of length 0 between different nodes.")
  if k == 1:
    return [edge for edge in edges if edge[0] == source and edge[1] == sink]
  paths = []
  from_source = [edge for edge in edges if edge[0] == source]
  for edge in from_source:
    current_path = [source]
    neighbour = edge[1]
    if neighbour == sink:
      paths.append(current_path + [sink])
    else:
      from_neighbourhood = paths_k(edges, k - 1, neighbour, sink)
      for p in from_neighbourhood:
        if len(current_path) + len(p) - 1 <= k:
          current_path += p
          paths.append(current_path)
  return paths

# For simplicity, consider just the edges and ignore their weights.
my_edges = edges_threshold.keys()
ss_path = paths_k(my_edges, 6, "Hattrem", "Kingambit")
print(f"Paths of Length 6 from Hattrem to Kingambit: {ss_path}")
print(f"Number of Paths of Length 6 from Hattrem to Kingambit: {len(ss_path)}")

## Exercise 5

Find the heaviest $3$-subgraph, which is a graph containing $k = 3$ nodes such that the sum on the remaining edges is maximum.

In [ ]:
def path_length(path, edges):
  total = 0
  for i in range(2):
    start_point = path[i]
    end_point = path[i + 1]
    total += edges.get((start_point, end_point), 0)
  return total

heaviest_3 = None
maximum_3 = 0

for source in nodes_threshold:
  for sink in nodes_threshold:
    paths_3 = paths_k(edges_threshold.keys(), 2, source, sink)
    for p in paths_3:
      if len(p) == 3 and sorted(list(set(p))) == sorted(p):
        C = path_length(p, edges_threshold)
        if C > maximum_3:
          maximum_3 = C
          heaviest_3 = p

print(f"Heaviest 3-Subgraph: {heaviest_3}")
print(f"Value of the Heaviest 3-Subgraph: {C}")